In [ ]:
!pip install ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 33.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 47.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 36.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 80.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 62.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 66.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 68.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import cv2
import numpy as np
import json
import os
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/ec2-user/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
def run_inference(IMG_PATH, WEIGHTS_PATH, OUTPUT_IMG_PATH):
    PIXEL_SHRINK = 5
    MASK_ALPHA = 0.5
    COLOR_SEED = 42
    LABEL_MARGIN_PX = 2
    FONT_RATIO = 0.005
    FONT_MIN = 8
    FONT_MAX = 32
    BOX_THICK_RATIO = 0.005
    MASK_THICK_RATIO = 0.0025
    THICK_MIN = 1
    THICK_MAX = 20
    FONT_FACE = cv2.FONT_HERSHEY_SIMPLEX
    TEXT_THICKNESS = 3

    model = YOLO(WEIGHTS_PATH)
    results = model(IMG_PATH, retina_masks=True)
    r = results[0]

    img = r.orig_img.copy()
    H, W = img.shape[:2]

    boxes = r.boxes.xyxy.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy().astype(int)
    scores = r.boxes.conf.cpu().numpy()
    names = r.names
    masks = r.masks.data.cpu().numpy() if r.masks is not None else []

    rng = np.random.default_rng(COLOR_SEED)
    colors = [tuple(int(c) for c in rng.integers(0, 256, 3)) for _ in range(len(boxes))]

    # Erode masks
    eroded_masks = []
    if len(masks) > 0:
        kernel = np.ones((2 * PIXEL_SHRINK + 1, 2 * PIXEL_SHRINK + 1), np.uint8)
        for m in masks:
            bm = (m * 255).astype(np.uint8)
            ed = cv2.erode(bm, kernel, iterations=1)
            eroded_masks.append((ed > 0).astype(np.uint8))
        eroded_masks = np.stack(eroded_masks, axis=0)
        composite_mask = np.any(eroded_masks, axis=0)
    else:
        composite_mask = np.zeros((H, W), dtype=bool)
        eroded_masks = []

    def shrink_box(b, d):
        x1, y1, x2, y2 = b
        return [x1 + d, y1 + d, x2 - d, y2 - d]

    shrunk_boxes = np.array([shrink_box(b, PIXEL_SHRINK) for b in boxes], dtype=int)

    font_size = int(np.clip(H * FONT_RATIO, FONT_MIN, FONT_MAX))
    box_thick = int(np.clip(H * BOX_THICK_RATIO, THICK_MIN, THICK_MAX))
    mask_thick = int(np.clip(H * MASK_THICK_RATIO, THICK_MIN, THICK_MAX))

    for idx, em in enumerate(eroded_masks):
        color = colors[idx]
        mask_bool = em.astype(bool)
        img[mask_bool] = (
            img[mask_bool] * (1 - MASK_ALPHA) + np.array(color) * MASK_ALPHA
        ).astype(np.uint8)
        cnts, _ = cv2.findContours((em * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cnts, -1, color, thickness=mask_thick, lineType=cv2.LINE_AA)

    annotator = Annotator(img, line_width=box_thick, font_size=font_size)
    label_rects = []

    for idx, (box, cls, conf) in enumerate(zip(shrunk_boxes, classes, scores)):
        annotator.box_label(box.tolist(), '', color=colors[idx])
        label = f"{names[int(cls)]} {conf:.2f}"
        (txt_w, txt_h), baseline = cv2.getTextSize(label, FONT_FACE, font_size / 10, thickness=TEXT_THICKNESS)
        lx, ly = box[0], box[1] - LABEL_MARGIN_PX
        if ly - txt_h - baseline < 0:
            ly = box[3] + txt_h + LABEL_MARGIN_PX
        rect = [lx, ly - txt_h - baseline, lx + txt_w, ly]

        def overlaps_any(r, prev_rects):
            for p in prev_rects:
                if not (r[2] < p[0] or r[0] > p[2] or r[3] < p[1] or r[1] > p[3]):
                    return True
            x0, y0, x1, y1 = r
            return np.any(composite_mask[y0:y1, x0:x1])

        while overlaps_any(rect, label_rects):
            rect[1] += txt_h + baseline + LABEL_MARGIN_PX
            rect[3] += txt_h + baseline + LABEL_MARGIN_PX

        label_rects.append(rect)

        cv2.rectangle(img, (rect[0], rect[1]), (rect[2], rect[3] + baseline), colors[idx], thickness=cv2.FILLED)
        cv2.putText(img, label, (rect[0], rect[3]), FONT_FACE, font_size / 10, (255, 255, 255), thickness=TEXT_THICKNESS, lineType=cv2.LINE_AA)

    out = annotator.result()
    cv2.imwrite(OUTPUT_IMG_PATH, out)
    print(f"Saved annotated image to {OUTPUT_IMG_PATH}")

    json_list = json.loads(r.to_json(normalize=False, decimals=5))
    minimal_list = [
        {
            "name": det["name"],
            "class": det["class"],
            "confidence": det["confidence"],
            "box": det["box"]
        } for det in json_list
    ]

    json_path = os.path.splitext(OUTPUT_IMG_PATH)[0] + ".json"
    with open(json_path, "w") as f:
        json.dump(minimal_list, f, indent=4)

    print(f"Saved JSON to {json_path}")
    return minimal_list

In [8]:
IMG_PATH = "16.JPG"
WEIGHTS_PATH = "runs/segment/train/weights/last.pt"
OUTPUT_IMG_PATH = "16_output.jpg"

output_json = run_inference(IMG_PATH, WEIGHTS_PATH, OUTPUT_IMG_PATH)
print(json.dumps(output_json, indent=4))


image 1/1 /home/ec2-user/SageMaker/16.JPG: 864x640 1 Fate-Line, 1 Head-Line, 1 Heart-Line, 1 Life-Line, 10.1ms
Speed: 4.6ms preprocess, 10.1ms inference, 8.5ms postprocess per image at shape (1, 3, 864, 640)
Saved annotated image to 16_output.jpg
Saved JSON to 16_output.json
[
    {
        "name": "Head-Line",
        "class": 1,
        "confidence": 0.89829,
        "box": {
            "x1": 378.04865,
            "y1": 2345.56885,
            "x2": 1594.03015,
            "y2": 2948.23853
        }
    },
    {
        "name": "Life-Line",
        "class": 3,
        "confidence": 0.89787,
        "box": {
            "x1": 811.00555,
            "y1": 2327.30444,
            "x2": 1619.44714,
            "y2": 3274.25415
        }
    },
    {
        "name": "Heart-Line",
        "class": 2,
        "confidence": 0.89529,
        "box": {
            "x1": 100.86046,
            "y1": 2271.27368,
            "x2": 1109.41357,
            "y2": 2618.68579
        }
    },
    {
